In [11]:
import random
from pathlib import Path
from cogent3 import load_aligned_seqs

# Set the random seed for reproducibility
random.seed(42)
sample_size = 10

# Point to the correct folder
input_dir = Path("./data/mammals")

# Get all files matching HMP_*.fa
all_fasta_files = sorted(input_dir.glob("HMP_*.fa"))

# Randomly sample 10 files
sampled_files = random.sample(all_fasta_files, sample_size)

# Initialize dictionary to hold ortholog data
orthologs = {}

for file in sampled_files:
    ensembl_id = file.stem.replace("HMP_", "")
    aln = load_aligned_seqs(file, moltype="dna")

    sequences = {}
    for name, seq in aln.named_seqs.items():
        lowered = name.lower()
        if "homo sapiens" in lowered:
            sequences["homo sapiens"] = str(seq)
        elif "pan troglodytes" in lowered:
            sequences["pan troglodytes"] = str(seq)
        elif "mus musculus" in lowered:
            sequences["mus musculus"] = str(seq)

    if "homo sapiens" in sequences and "pan troglodytes" in sequences:
        orthologs[ensembl_id] = sequences


In [ ]:
from cogent3 import make_unaligned_seqs
from cogent3.align.align import global_pairwise, make_dna_scoring_dict
import time
# Setup the DNA scoring matrix
score_matrix = make_dna_scoring_dict(match=5, transition=-2, transversion=-4)
gap_open = 4
gap_extend = 1

results = {}

for ensembl_id, seqs in orthologs.items():
    human = seqs.get("homo sapiens")
    chimp = seqs.get("pan troglodytes")
    mouse = seqs.get("mus musculus")

    pairwise_results = {}
    pairs = {"human_chimp": (human, chimp)}
    if mouse:
        pairs["human_mouse"] = (human, mouse)

    for pair_name, (seq1, seq2) in pairs.items():
        # Create SequenceCollection with degapped sequences
        start = time.time()
        unaligned = make_unaligned_seqs(
            data={"seq1": seq1.replace("-", ""), "seq2": seq2.replace("-", "")},
            moltype="dna"
        )

        # Extract raw cogent3 Sequence objects
        s1, s2 = unaligned.seqs

        # Time the alignment
        aln, score = global_pairwise(
            s1, s2, score_matrix,
            gap_open, gap_extend,
            return_score=True
        )
        elapsed = time.time() - start

        # Store result
        pairwise_results[pair_name] = {
            "alignment": aln.to_dict(),
            "score": score,
            "time_seconds": elapsed
        }

    results[ensembl_id] = pairwise_results

example_id = next(iter(results))
print(f"Example: {example_id}")
pprint(results[example_id])
example_id = next(iter(results))
print(f"Example: {example_id}")
pprint(results[example_id])



Example: ENSG00000182749
{'human_chimp': {'alignment': {'seq1': 'TCACTTGGTCTTCTGATCAAGTTTGCGCTGTACCAGCTGGCTCAGGAGGAATGCAGTGAGGATGCTGCTGCCCACCGTGAGCAGGAAGAGGCCAGAAAAGTTGTGAGGCCAGTGCGTGTGCAGAGGCTCATAGATGGGCCGTCGGGCCTCATAGTCCAGTGCCACAGCCTCCAGCTGAGCCAGCGTGCACAGCACCAAGAAGATGTGGAAAAGTTGGTGGCCCTGCCCGAAGACATGGCAGCTGCCAGGGAACCAGCGCTCGGGCATGAAGGTAGAGAAGAAGGCAGCAGCCAGCAGAAAGAAGACCACCTGGCACTTGTGGTAGAGAAGAGCTGGATCATCCGTGGTGGGGTCGGAGGACACGAAGATACGATGCACCACAGGACTAATGTCCAGTGCGTAGGCCAGGACGGAGGGCACCTCCTGGCATGTGCGGCCCAGCAGGCCTGGTTTCTGGATGTACTTGTTATAGCAGGAGCCAATGCAGGAAAGCCAGGCGAGAAAGGCAGCCATGGGCAGAAAAACAGCCTGCACCTGGGCATGCCAGGCGGGCTCGATAGCATAGTAGAAGTGTGCCAAGGCACTGCCAAACTGGTACACGGCCACCCCCACATAGTCCAGGAAGAAGAAGCTGTAATGCCAGAACTCAGACTTGGCCTGCAGGAGGTGAGCCAAGGCACTGAAGGAGAGGTAGGTGAAAGAGGCAAGGACAATGATGAAGAGGGGCAGGGCGTGTGGGTCTCCCCAGAAGTCCACGGTCTCCACAAAGAGGGCCAGCCGCAGCAGCAGTACCAGGGCCGCCAGCAGGTGGGTCCAGACATTCACGGCCTCGTTGTGCTGCTGGAACAGCGTGCGGAAATAGAAGCGCCAGGTCTGATGCAGCGGCCGGTAGCCCGCATAGATGTACGGCTTCCAGAAGAGCGGCGGCACCTCAG

In [14]:
for pair_name, (seq1, seq2) in pairs.items():
    unaligned = make_unaligned_seqs(
        data={"seq1": seq1.replace("-", ""), "seq2": seq2.replace("-", "")},
        moltype="dna"
    )

    s1, s2 = unaligned.seqs

    # Align with Cogent3
    start = time.time()
    aln, score = global_pairwise(s1, s2, score_matrix, gap_open, gap_extend, return_score=True)
    elapsed = time.time() - start

    cogent_aln = aln.to_dict()
    cogent_seq1 = cogent_aln["seq1"]
    cogent_seq2 = cogent_aln["seq2"]

    # Placeholder: MADB uses same alignment, but perturbed time
    madb_time = elapsed + random.uniform(-0.05, 0.05)  # ±50ms jitter

    # Compute identity between alignments (this will be 1.0 since they're identical for now)
    def percent_identity(s1, s2):
        matches = sum(a == b for a, b in zip(s1, s2) if a != '-' and b != '-')
        length = sum(1 for a, b in zip(s1, s2) if a != '-' and b != '-')
        return matches / length if length else 0.0

    alignment_identity = percent_identity(cogent_seq1, cogent_seq2)

    pairwise_results[pair_name] = {
        "cogent3": {
            "alignment": cogent_aln,
            "score": score,
            "time_seconds": elapsed,
        },
        "madb": {
            "alignment": cogent_aln,  # using same alignment for now
            "time_seconds": madb_time,
        },
        "identity_score": alignment_identity
    }


In [ ]:
example_id = next(iter(pairwise_results))
print(f"Example: {example_id}")
pprint(pairwise_results[example_id])

Example: human_chimp
{'cogent3': {'alignment': {'seq1': 'TTAATCCTCGTCTTCCTC-CTCTTCTTCGTCCTGGTTAATCTGGAAGTAACGTAATTCGT-AACTCTCTTTGCTGTTAGCAACTACGCGCAACCAGTCACGTAGATTATTCTTCTTCAAATATTTTTTGGTGAGATATTTCAAATACCTTTTGGAGAAAGGCACCTCGGATGTCACGGTGATCTTGCTCTTGCTCCTTTCGATGGTCACCACCCCTCCACCAAGGTTCCCAGCTTTTCCGTTCACTTTGATCCTTTCTTGCAAAAACTGCTCAAAATTGGCAGCATCCATGATTCCATCTTCTACAGGGTGGGTGCAATCAAGAGTGAACTTCAGAACTTGCTTCTTTTTTTTGCCCCCCTTCACCACAAGCTTTTTCACAGGAGCCAT',
                           'seq2': 'TCA---------TTCCTCACACT----CGTC------A-----GA-G----GT--TT-GTGAACA------GCC---AGC--CCA-GAGCAA--AGCC-C--AGAGGG--------CAG---------GGCGGGGGA----GGAGAC-TTTTGCAGAAAGTCACCTTGGATGTCACAGTGATCTTGCTCTTGCTCCTTTCGTTGGTCACCACCCCTCCACCAAGGTTCCCAGCTTTTCCGTTCACTTTGATCCTTTCTTGCAAAAACCGCTCAAAATTGGCAGCATCCAT---------------------------------------------------------------------------------------------------'},
             'score': 898.4040660893911,
             'time_seconds': 0.010627508163452148},
 'identity_score': 0.88940092

In [5]:
import pandas as pd

records = []

for ensembl_id, pairs in results.items():
    for pair_name, data in pairs.items():
        if "cogent3" not in data or "madb" not in data:
            continue

        # Performance (time)
        records.append({
            "tool": "cogent3",
            "pair": pair_name,
            "metric": "time_seconds",
            "value": data["cogent3"]["time_seconds"]
        })
        records.append({
            "tool": "madb",
            "pair": pair_name,
            "metric": "time_seconds",
            "value": data["madb"]["time_seconds"]
        })

        # Accuracy (alignment identity)
        records.append({
            "tool": "cogent3",
            "pair": pair_name,
            "metric": "identity_score",
            "value": data["identity_score"]
        })
        records.append({
            "tool": "madb",
            "pair": pair_name,
            "metric": "identity_score",
            "value": data["identity_score"]
        })

df = pd.DataFrame(records)


In [6]:
import plotly.express as px

# First: Performance violin plot
fig_perf = px.violin(
    df[df["metric"] == "time_seconds"],
    x="pair",
    y="value",
    color="tool",
    box=True,
    points="all",
    title="Alignment Runtime (seconds)",
    labels={"value": "Time (s)", "pair": "Species Pair"},
)
fig_perf.update_layout(yaxis_title="Time (s)", xaxis_title="Pair")
fig_perf.show()

# Second: Accuracy violin plot
fig_acc = px.violin(
    df[df["metric"] == "identity_score"],
    x="pair",
    y="value",
    color="tool",
    box=True,
    points="all",
    title="Alignment Identity Score",
    labels={"value": "Identity", "pair": "Species Pair"},
)
fig_acc.update_layout(yaxis_title="Percent Identity", xaxis_title="Pair")
fig_acc.show()
